# Best FT-CNN Model with Grad-CAM

This notebook keeps the original `best_ft_cnn_model.ipynb` workflow, but updates it so the **model input is an explicit two-channel tensor**:

- channel 0 = spatial grayscale image
- channel 1 = frequency-domain log-magnitude FFT image

Grad-CAM is then computed directly on the trained two-channel model.


In [ ]:
import os
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    RandomFlip,
    RandomRotation,
    RandomZoom,
    RandomContrast,
)
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

AUTOTUNE = tf.data.AUTOTUNE
DATA_DIR = "dataset"
CLASS_NAMES = ['F0', 'F1', 'F2', 'F3', 'F4']
INPUT_SIZE = (256, 256)
LAST_CONV_LAYER_NAME = "gradcam_target_conv"

os.listdir(DATA_DIR)


In [ ]:
data = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    color_mode='grayscale',
    batch_size=32,
    image_size=INPUT_SIZE,
    class_names=CLASS_NAMES,
)

data = data.map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)
print(f"Total batches: {len(data)}")

train_size = int(len(data) * 0.7)
val_size = int(len(data) * 0.15) + 1
test_size = int(len(data) * 0.15) + 1

train = data.take(train_size)
val = data.skip(train_size).take(val_size)
test = data.skip(train_size + val_size).take(test_size)

print(train_size, val_size, test_size)


In [ ]:
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.027),
    RandomZoom(0.1),
    RandomContrast(0.1),
])


def augment(image, label):
    return data_augmentation(image, training=True), label


def perform_fft_processing(input_tensor):
    x = tf.squeeze(input_tensor, axis=-1)
    x = tf.cast(x, tf.complex64)
    x = tf.signal.fft2d(x)
    x = tf.signal.fftshift(x, axes=(1, 2))
    x = tf.abs(x)
    x = tf.math.log(1.0 + x)
    x = tf.expand_dims(x, axis=-1)
    return x


def stack_spatial_and_frequency_channels(image, label):
    fft_image = perform_fft_processing(image)
    two_channel_image = tf.concat([image, fft_image], axis=-1)
    return two_channel_image, label


train_final = (
    train
    .map(augment, num_parallel_calls=AUTOTUNE)
    .map(stack_spatial_and_frequency_channels, num_parallel_calls=AUTOTUNE)
    .prefetch(buffer_size=AUTOTUNE)
)

val_final = (
    val
    .map(stack_spatial_and_frequency_channels, num_parallel_calls=AUTOTUNE)
    .prefetch(buffer_size=AUTOTUNE)
)

test_final = (
    test
    .map(stack_spatial_and_frequency_channels, num_parallel_calls=AUTOTUNE)
    .prefetch(buffer_size=AUTOTUNE)
)

sample_batch, sample_labels = next(iter(train_final))
print("Two-channel batch shape:", sample_batch.shape)
print("Sample labels shape:", sample_labels.shape)


In [ ]:
def create_combined_model(input_shape=(256, 256, 2), num_classes=5):
    inputs = Input(shape=input_shape, name="spatial_frequency_input")

    x = Conv2D(16, (3, 3), activation='relu', name='conv_1')(inputs)
    x = MaxPooling2D(name='pool_1')(x)
    x = Conv2D(32, (3, 3), activation='relu', name='conv_2')(x)
    x = MaxPooling2D(name='pool_2')(x)
    x = Conv2D(16, (3, 3), activation='relu', name=LAST_CONV_LAYER_NAME)(x)
    x = MaxPooling2D(name='pool_3')(x)

    x = Flatten(name='flatten_features')(x)
    x = Dense(256, activation='relu', name='dense_features')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)

    return Model(inputs=inputs, outputs=outputs, name='spatial_spectral_fft_cnn')


model = create_combined_model(input_shape=(256, 256, 2), num_classes=len(CLASS_NAMES))
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()


In [ ]:
log_dir = 'logs'
callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    TensorBoard(log_dir=log_dir),
]

history = model.fit(
    train_final,
    validation_data=val_final,
    epochs=50,
    callbacks=callbacks,
)


In [ ]:
test_loss, test_accuracy = model.evaluate(test_final)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


## Grad-CAM helpers

The functions below assume the input sample already has shape `(H, W, 2)` where the last dimension contains `[spatial, frequency]`.


In [ ]:
def make_gradcam_heatmap(input_tensor, model, layer_name=LAST_CONV_LAYER_NAME, class_index=None):
    if input_tensor.ndim == 3:
        input_tensor = tf.expand_dims(input_tensor, axis=0)

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(input_tensor, training=False)
        if class_index is None:
            class_index = tf.argmax(predictions[0])
        class_score = predictions[:, class_index]

    gradients = tape.gradient(class_score, conv_outputs)
    pooled_gradients = tf.reduce_mean(gradients, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_gradients, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())
    return heatmap.numpy(), int(class_index), predictions[0].numpy()


def overlay_gradcam(spatial_image, heatmap, alpha=0.4):
    spatial_image = np.squeeze(spatial_image)
    heatmap = tf.image.resize(heatmap[..., np.newaxis], spatial_image.shape, method='bilinear').numpy()
    heatmap = np.squeeze(heatmap)

    spatial_min = np.min(spatial_image)
    spatial_max = np.max(spatial_image)
    spatial_norm = (spatial_image - spatial_min) / (spatial_max - spatial_min + 1e-8)

    overlay = np.stack([spatial_norm, spatial_norm, spatial_norm], axis=-1)
    overlay[..., 0] = np.clip(overlay[..., 0] + alpha * heatmap, 0, 1)
    overlay[..., 1] = np.clip(overlay[..., 1] * (1 - 0.5 * alpha), 0, 1)
    overlay[..., 2] = np.clip(overlay[..., 2] * (1 - alpha), 0, 1)
    return overlay


In [ ]:
for batch_images, batch_labels in test_final.take(1):
    sample_image = batch_images[0]
    sample_label = int(batch_labels[0].numpy())
    break

heatmap, predicted_index, prediction_scores = make_gradcam_heatmap(sample_image, model)
overlay = overlay_gradcam(sample_image[..., 0], heatmap)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(sample_image[..., 0], cmap='gray')
plt.title(f"Spatial channel\nTrue label: {CLASS_NAMES[sample_label]}")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(sample_image[..., 1], cmap='magma')
plt.title('Frequency channel')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title(
    f"Grad-CAM overlay\nPredicted: {CLASS_NAMES[predicted_index]}\nScore: {prediction_scores[predicted_index]:.3f}"
)
plt.axis('off')

plt.tight_layout()
plt.show()
